# Get the data: one Census table for the whole project

**What this does:** Downloads every Census number the project needs, for each ZIP code area I serve, and saves it as one table: `data/census_by_zip.csv`.

**Why:** Each question in this project gets its own short notebook. They all read this one table, so the download code lives in one place and every notebook uses the same numbers.

**Data:** U.S. Census Bureau American Community Survey (ACS) 5-year estimates, 2020-2024. Public and free.

**How to run:** Save a free Census API key in `census_key.txt` in the project folder (get one at https://api.census.gov/data/key_signup.html). Then run the cells top to bottom. You only need to run this again to add ZIP codes, add variables, or move to a newer year.

In [14]:
import os, re
import requests
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

# The key file is listed in .gitignore, so it never goes to GitHub. Never paste the key here.
API_KEY = os.environ.get("CENSUS_API_KEY")
KEY_FILE = "../census_key.txt"
if not API_KEY and os.path.exists(KEY_FILE):
    API_KEY = open(KEY_FILE, encoding="utf-8-sig").read().strip()
print("Using an API key:", bool(API_KEY))

Using an API key: True


## 1. The areas I serve

ZIP code areas do not match city borders exactly (94608 includes Emeryville, 94610 and 94611 include parts of Piedmont). The neighborhood names are my own approximate labels, not Census names.

In [15]:
YEAR = 2024  # ACS 5-year estimates covering 2020-2024
BASE = f"https://api.census.gov/data/{YEAR}/acs/acs5"

# ZIP code: (city, approximate neighborhood name)
AREAS = {
    "94501": ("Alameda",  "Alameda, main island"),
    "94502": ("Alameda",  "Alameda, Bay Farm Island"),
    "94601": ("Oakland",  "Oakland, Fruitvale"),
    "94602": ("Oakland",  "Oakland, Glenview / Dimond"),
    "94603": ("Oakland",  "Oakland, Elmhurst"),
    "94605": ("Oakland",  "Oakland, East Oakland hills"),
    "94606": ("Oakland",  "Oakland, San Antonio / Eastlake"),
    "94607": ("Oakland",  "Oakland, West Oakland / Chinatown"),
    "94608": ("Oakland",  "Emeryville / North Oakland"),
    "94609": ("Oakland",  "Oakland, Temescal"),
    "94610": ("Oakland",  "Oakland, Grand Lake"),
    "94611": ("Oakland",  "Oakland, Montclair / Piedmont"),
    "94612": ("Oakland",  "Oakland, downtown"),
    "94618": ("Oakland",  "Oakland, Rockridge"),
    "94619": ("Oakland",  "Oakland, Laurel / Redwood Heights"),
    "94621": ("Oakland",  "Oakland, Coliseum area"),
    "94702": ("Berkeley", "Berkeley, west"),
    "94703": ("Berkeley", "Berkeley, central / south"),
    "94704": ("Berkeley", "Berkeley, downtown / campus"),
    "94705": ("Berkeley", "Berkeley, Claremont / Elmwood"),
    "94707": ("Berkeley", "Berkeley, north hills"),
    "94708": ("Berkeley", "Berkeley hills"),
    "94709": ("Berkeley", "Berkeley, north"),
    "94710": ("Berkeley", "Berkeley, waterfront"),
}
all_zips = list(AREAS)
print(len(all_zips), "ZIP code areas")

24 ZIP code areas


## 2. Choose the Census variables by their labels

Census variables have codes like `B01001_020E` that mean nothing by themselves. This cell downloads the Census's own list of labels and picks variables by their **label text**. Each column of my table is the sum of one or more Census variables. The cell checks that it found the number of variables it expected, and stops with a clear message if not.

In [16]:
def get(url, params=None):
    params = dict(params or {})
    if API_KEY:
        params["key"] = API_KEY
    r = requests.get(url, params=params, timeout=120)
    if r.status_code != 200:
        raise RuntimeError(f"Census API answered {r.status_code}: {r.text[:300]}")
    try:
        return r.json()
    except ValueError:
        title = re.search(r"<title>(.*?)</title>", r.text, re.S)
        title = title.group(1).strip() if title else r.text[:200]
        if "key" in title.lower() or not API_KEY:
            raise PermissionError(
                f"The Census did not send data (it said: {title!r}). Get a free key at "
                "https://api.census.gov/data/key_signup.html, save it in census_key.txt "
                "in the project folder, then run all cells again.")
        raise RuntimeError(f"Census API sent something that is not data: {title!r}")

meta = get(BASE + "/variables.json")["variables"]
print(len(meta), "variables available from the Census")

def pick(table, contains=(), ends=(), excludes=()):
    # Codes in one Census table whose label has every text in `contains`,
    # ends with one of the texts in `ends` (if given), and has none of `excludes`.
    found = []
    for code, info in meta.items():
        if not code.startswith(table + "_") or not code.endswith("E"):
            continue
        label = info.get("label", "")
        if not all(c in label for c in contains):
            continue
        if ends and not any(label.endswith(e) for e in ends):
            continue
        if any(x in label for x in excludes):
            continue
        found.append(code)
    return sorted(found)

AGES_65 = ("65 and 66 years", "67 to 69 years", "70 to 74 years", "75 to 79 years", "80 to 84 years", "85 years and over")
AGES_75 = AGES_65[3:]
OLDER = ("65 to 74 years", "75 to 84 years", "85 years and over")       # householder age groups
NOT_VERY_WELL = ('"well"', '"not well"', '"not at all"')                  # speaks English less than "very well"

# column name: (plain description, Census codes, how many codes I expect)
SPEC = {
    "residents_65plus":   ("People age 65 or older", pick("B01001", ends=AGES_65), 12),
    "residents_75plus":   ("People age 75 or older", pick("B01001", ends=AGES_75), 6),
    "residents_85plus":   ("People age 85 or older", pick("B01001", ends=AGES_65[5:]), 2),
    "live_alone_65plus":  ("People 65+ who live alone", pick("B09020", ends=("!!Living alone",)), 2),
    "live_alone_men_65plus":   ("Men 65+ who live alone", pick("B09020", contains=["Male:"], ends=("!!Living alone",)), 1),
    "live_alone_women_65plus": ("Women 65+ who live alone", pick("B09020", contains=["Female:"], ends=("!!Living alone",)), 1),
    "group_quarters_65plus": ("People 65+ living in group quarters, mostly nursing homes (not in a household)",
                           pick("B09020", ends=("In group quarters",)), 1),
    "selfcare_65plus":    ("People 65+ with difficulty bathing or dressing",
                           pick("B18106", contains=["With a self-care difficulty"], excludes=["5 to 17", "18 to 34", "35 to 64"]), 4),
    "indep_living_difficulty_65plus": ("People 65+ with difficulty doing errands alone, such as shopping or a doctor visit",
                           pick("B18107", contains=["With an independent living difficulty"], excludes=["18 to 34", "35 to 64"]), 4),
    "hh65_total":         ("Households headed by someone 65+", pick("B19037", ends=("Householder 65 years and over:",)), 1),
    "median_income_65plus_hh": ("Median yearly income of households headed by someone 65+ (dollars)",
                           pick("B19049", ends=("Householder 65 years and over",)), 1),
    "median_income_men_alone_65plus": ("Median yearly income of men 65+ living alone (dollars)",
                           pick("B19215", contains=["Male householder", "!!Living alone", "65 years and over"]), 1),
    "median_income_women_alone_65plus": ("Median yearly income of women 65+ living alone (dollars)",
                           pick("B19215", contains=["Female householder", "!!Living alone", "65 years and over"]), 1),
    "owner_hh65":         ("Households headed by someone 65+ that own their home",
                           pick("B25007", contains=["Owner occupied"], ends=tuple("Householder " + a for a in OLDER)), 3),
    "renter_hh65":        ("Households headed by someone 65+ that rent",
                           pick("B25007", contains=["Renter occupied"], ends=tuple("Householder " + a for a in OLDER)), 3),
    "lang65_total":       ("People 65+ counted in the language table", pick("B16004", ends=("!!65 years and over:",)), 1),
    "english_only_65plus": ("People 65+ who speak only English at home",
                           pick("B16004", contains=["65 years and over"], ends=("Speak only English",)), 1),
    "spanish_65plus":     ("People 65+ who speak Spanish at home",
                           pick("B16004", contains=["65 years and over"], ends=("Speak Spanish:",)), 1),
    "spanish_limited_english_65plus": ("People 65+ who speak Spanish at home and speak English less than very well",
                           pick("B16004", contains=["65 years and over", "Speak Spanish:"], ends=NOT_VERY_WELL), 3),
    "asian_pacific_65plus": ("People 65+ who speak an Asian or Pacific Island language at home",
                           pick("B16004", contains=["65 years and over"], ends=("Speak Asian and Pacific Island languages:",)), 1),
    "asian_pacific_limited_english_65plus": ("People 65+ who speak an Asian or Pacific Island language at home and speak English less than very well",
                           pick("B16004", contains=["65 years and over", "Speak Asian and Pacific Island languages:"], ends=NOT_VERY_WELL), 3),
    "hh65_no_vehicle":    ("Households headed by someone 65+ with no car",
                           pick("B25045", contains=["No vehicle available"], ends=("Householder 65 years and over",)), 2),
    "hh65_with_vehicle":  ("Households headed by someone 65+ with at least one car",
                           pick("B25045", contains=["1 or more vehicles available"], ends=("Householder 65 years and over",)), 2),
    "tech65_total":       ("People 65+ counted in the computer and internet table", pick("B28005", ends=("!!65 years and over:",)), 1),
    "no_computer_65plus": ("People 65+ in a home with no computer",
                           pick("B28005", contains=["65 years and over"], ends=("No computer",)), 1),
    "computer_no_internet_65plus": ("People 65+ in a home with a computer but no internet subscription",
                           pick("B28005", contains=["65 years and over"], ends=("Without an Internet subscription",)), 1),
    "broadband_65plus":   ("People 65+ in a home with a computer and broadband internet",
                           pick("B28005", contains=["65 years and over"], ends=("With a broadband Internet subscription",)), 1),
}

# Income brackets for households headed by someone 65+: one column per bracket
def lower_bound(label):
    last = label.split("!!")[-1]
    if last.startswith("Less than"):
        return 0
    return int(re.search(r"\$([\d,]+)", last).group(1).replace(",", ""))

bracket_codes = pick("B19037", contains=["Householder 65 years and over:!!"])
for code in bracket_codes:
    lo = lower_bound(meta[code]["label"])
    SPEC[f"hh65_income_from_{lo // 1000}k"] = (
        "Households headed by someone 65+ with yearly income: " + meta[code]["label"].split("!!")[-1], [code], 1)
assert len(bracket_codes) == 16, f"Expected 16 income brackets, found {len(bracket_codes)}."

problems = [f"{name}: expected {n} Census variables, found {len(codes)}" for name, (_, codes, n) in SPEC.items() if len(codes) != n]
assert not problems, "Some picks did not match the Census labels:\n" + "\n".join(problems)
print(len(SPEC), "columns built from", sum(len(c) for _, c, _ in SPEC.values()), "Census variables. All picks matched.")

28475 variables available from the Census
43 columns built from 77 Census variables. All picks matched.


## 3. Download the numbers

In [17]:
codes = sorted({c for _, cs, _ in SPEC.values() for c in cs})

def pull(codes, zips, chunk=40):
    # The API allows about 50 variables per request, so ask in chunks.
    # If the Census rejects my list of ZIP codes, ask for every ZIP area and keep mine.
    frames = []
    for i in range(0, len(codes), chunk):
        part = codes[i:i + chunk]
        ask = {"get": "NAME," + ",".join(part)}
        try:
            rows = get(BASE, {**ask, "for": "zip code tabulation area:" + ",".join(zips)})
        except RuntimeError as err:
            print("Specific ZIP list failed, asking for all ZIP areas instead:", str(err)[:120])
            rows = get(BASE, {**ask, "for": "zip code tabulation area:*"})
        df = pd.DataFrame(rows[1:], columns=rows[0])
        df = df.rename(columns={"zip code tabulation area": "zip"}).set_index("zip")
        df = df[df.index.isin(zips)]
        frames.append(df[part].apply(pd.to_numeric, errors="coerce"))
    return pd.concat(frames, axis=1)

raw = pull(codes, all_zips)
raw = raw.where(raw >= 0)  # the Census uses large negative numbers for 'not available'
missing = [z for z in all_zips if z not in raw.index]
print(raw.shape[0], "ZIP areas and", raw.shape[1], "Census variables downloaded")
if missing:
    print("Not returned by the Census, left out:", missing)

24 ZIP areas and 67 Census variables downloaded


## 4. Build the table, check it, and save it

Three files go into the `data` folder:

- `census_by_zip.csv`: the table. One row per ZIP code area, one column per measure.
- `data_dictionary.csv`: what each column means, in plain words.
- `variable_labels.csv`: which Census variables went into each column, for anyone who wants to check my work.

In [18]:
table = pd.DataFrame(index=raw.index)
table["city"] = table.index.map(lambda z: AREAS[z][0])
table["area"] = table.index.map(lambda z: AREAS[z][1])
for name, (_, cs, _) in SPEC.items():
    # min_count=1 keeps a value blank when the Census published nothing, instead of showing 0
    table[name] = raw[cs].sum(axis=1, min_count=1)
table = table.sort_index()

# Checks: the pieces should add up
bracket_cols = [c for c in table.columns if c.startswith("hh65_income_from_")]
gap = (table[bracket_cols].sum(axis=1) - table["hh65_total"]).abs().max()
assert gap <= 1, f"Income brackets do not add up to the household total (off by {gap})."
assert (table["residents_75plus"] <= table["residents_65plus"]).all()
assert (table["residents_85plus"] <= table["residents_75plus"]).all()
assert (table["spanish_limited_english_65plus"] <= table["spanish_65plus"]).all()
assert (table["group_quarters_65plus"] <= table["residents_65plus"]).all()
assert (table["live_alone_men_65plus"] + table["live_alone_women_65plus"] == table["live_alone_65plus"]).all()
blank = table.isna().sum()
print("Checks passed. Columns with blank values:", dict(blank[blank > 0]) or "none")

os.makedirs("../data", exist_ok=True)
table.to_csv("../data/census_by_zip.csv")
pd.DataFrame([(name, desc, len(cs)) for name, (desc, cs, _) in SPEC.items()],
             columns=["column", "meaning", "census_variables_used"]).to_csv("../data/data_dictionary.csv", index=False)
pd.DataFrame([(name, c, meta[c]["label"]) for name, (_, cs, _) in SPEC.items() for c in cs],
             columns=["column", "census_code", "census_label"]).to_csv("../data/variable_labels.csv", index=False)
print(f"Saved {table.shape[0]} rows and {table.shape[1]} columns to data/census_by_zip.csv")

Checks passed. Columns with blank values: {'median_income_men_alone_65plus': 4, 'median_income_women_alone_65plus': 1}
Saved 24 rows and 45 columns to data/census_by_zip.csv


In [19]:
# A quick look: totals across all my ZIP codes for the count columns
counts = [c for c in table.columns if c not in ("city", "area") and not c.startswith("median_")]
summary = pd.DataFrame({"meaning": {name: SPEC[name][0] for name in counts},
                        "total_all_areas": table[counts].sum().astype(int)})
summary

,meaning,total_all_areas
residents_65plus,People age 65 or older,103288
residents_75plus,People age 75 or older,42466
residents_85plus,People age 85 or older,12715
live_alone_65plus,People 65+ who live alone,28995
live_alone_men_65plus,Men 65+ who live alone,10538
live_alone_women_65plus,Women 65+ who live alone,18457
group_quarters_65plus,"People 65+ living in group quarters, mostly nu...",3298
selfcare_65plus,People 65+ with difficulty bathing or dressing,8249
indep_living_difficulty_65plus,People 65+ with difficulty doing errands alone...,14994
hh65_total,Households headed by someone 65+,64576


## How the other notebooks use this

Each question notebook starts with one line:

```python
table = pd.read_csv("../data/census_by_zip.csv", dtype={"zip": str}).set_index("zip")
```

### Limits to remember
- These are survey estimates. Small ZIP codes have wide margins of error, so treat small differences as ties.
- The Census publishes each measure separately. It does not say how many people are, for example, both 75+ and living alone. Combining columns means making assumptions.
- Median income columns can be blank for small ZIP codes where the Census does not publish a figure.